In [ ]:
# ============================================================
# PROGRAM 1 (FINAL FIX): RNN - Word-level Text Generation

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer

# --- Good sized word-level corpus ---
text = """
the weather today is sunny and warm
the weather tomorrow will be cloudy
it is raining heavily outside today
the sun is shining bright in the sky
cold winds are blowing from the north
the temperature is rising in summer
winter brings snow and cold weather
spring flowers bloom in warm weather
the sky is clear and blue today
storms bring thunder in the evening
warm sunny days are good for activities
the clouds are moving fast in the sky
today the wind is blowing very strongly
the morning is cold and the evening is warm
rain is falling softly in the garden
the sun rises early in the summer morning
clouds cover the sky in the afternoon
the weather is pleasant in the spring
snow falls gently during winter nights
the breeze is cool and refreshing today
""".lower().strip()

# --- Word-level Tokenization ---
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
vocab_size = len(tokenizer.word_index) + 1
sequences = tokenizer.texts_to_sequences([text])[0]
word2idx = tokenizer.word_index
idx2word = {v: k for k, v in word2idx.items()}

print(f"Vocabulary size  : {vocab_size} words")
print(f"Total tokens     : {len(sequences)}")

# --- Prepare Sequences (5-word context) ---
seq_len = 5
X, y = [], []
for i in range(len(sequences) - seq_len):
    X.append(sequences[i:i+seq_len])
    y.append(sequences[i+seq_len])

X = np.array(X)
y = to_categorical(np.array(y), num_classes=vocab_size)
print(f"Training samples : {len(X)}\n")

# --- Build Word-level RNN Model ---
model = Sequential([
    Embedding(vocab_size, 32, input_length=seq_len),
    SimpleRNN(64, activation='tanh'),
    Dense(64, activation='relu'),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# --- Train ---
history = model.fit(X, y, epochs=300, batch_size=16, verbose=0)

for ep in [100, 200, 300]:
    print(f"Epoch {ep} — Loss: {history.history['loss'][ep-1]:.4f} | Accuracy: {history.history['accuracy'][ep-1]:.4f}")

print(f"\n✅ Final Accuracy: {history.history['accuracy'][-1]:.4f}")

# --- Word-level Generator ---
def generate_text(seed_words, num_words=8, temperature=0.5):
    result = seed_words.lower().split()
    for _ in range(num_words):
        # take last seq_len words
        chunk = result[-seq_len:]
        # pad if shorter
        while len(chunk) < seq_len:
            chunk = ['the'] + chunk
        x = np.array([[word2idx.get(w, 1) for w in chunk]])

        pred = model.predict(x, verbose=0)[0]

        # temperature sampling
        pred = np.log(pred + 1e-10) / temperature
        pred = np.exp(pred) / np.sum(np.exp(pred))
        next_idx = np.random.choice(len(pred), p=pred)
        next_word = idx2word.get(next_idx, 'the')
        result.append(next_word)
    return ' '.join(result)

# --- Results ---
test_cases = [
    ("the weather today is", 0.3),
    ("the weather today is", 0.6),
    ("it is raining",        0.3),
    ("the sky is clear",     0.4),
    ("cold winds are",       0.3),
    ("the sun is",           0.5),
]

print("\n" + "="*65)
print("        RNN WORD-LEVEL TEXT GENERATION RESULTS")
print("="*65)
for seed, temp in test_cases:
    out = generate_text(seed, num_words=7, temperature=temp)
    print(f"\n🌡  Temp : {temp}")
    print(f"📥 Input : '{seed}'")
    print(f"📤 Output: '{out}'")
print("="*65)

In [ ]:
# ============================================================
# PROGRAM 2: LSTM - Sentiment Analysis
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Sample Dataset ---
sentences = [
    "This movie was absolutely fantastic", "I loved every moment of it",
    "Great film highly recommend", "Amazing acting and storyline",
    "The best movie I have ever seen", "Brilliant and entertaining",
    "This was a terrible movie", "I hated it completely boring",
    "Worst film ever waste of time", "Awful acting bad storyline",
    "I did not enjoy this at all", "Disappointing and dull"
]
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]  # 1=Positive, 0=Negative

# --- Tokenize ---
tokenizer = Tokenizer(num_words=500, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=10, padding='post', truncating='post')

X = np.array(padded)
y = np.array(labels)

# --- Build LSTM Model ---
model = Sequential([
    Embedding(500, 16, input_length=10),
    LSTM(32),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# --- Train ---
history = model.fit(X, y, epochs=50, verbose=0)
print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")

# --- Predict ---
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=10, padding='post')
    score = model.predict(pad, verbose=0)[0][0]
    label = "Positive" if score >= 0.5 else "Negative"
    return label, score

test_sentences = [
    "This movie was absolutely fantastic",
    "Worst film ever waste of time"
]
print("\n--- Predictions ---")
for s in test_sentences:
    label, score = predict_sentiment(s)
    print(f"Input   : '{s}'")
    print(f"Output  : {label} ({score:.4f})\n")

In [ ]:
# ============================================================
# PROGRAM 3: BERT - Text Classification using Hugging Face
# ============================================================

# Install required library
!pip install transformers -q

from transformers import pipeline

# --- Load BERT Sentiment Analysis Pipeline ---
print("Loading BERT model (bert-base-uncased fine-tuned)...")
classifier = pipeline(
    "text-classification",
    model="textattack/bert-base-uncased-imdb",
    tokenizer="textattack/bert-base-uncased-imdb"
)

# --- Test Sentences ---
test_sentences = [
    "I love this product, it works great!",
    "This is the worst experience I have ever had.",
    "The food was okay, nothing special.",
    "Absolutely brilliant performance by the entire cast!"
]

print("\n--- BERT Predictions ---")
print(f"{'Input':<50} {'Label':<12} {'Score':<8}")
print("-" * 75)
for sentence in test_sentences:
    result = classifier(sentence)[0]
    label = "POSITIVE" if result['label'] == 'LABEL_1' else "NEGATIVE"
    print(f"{sentence[:48]:<50} {label:<12} {result['score']:.4f}")

In [ ]:
# ============================================================
# PROGRAM 4: RoBERTa - Sentiment Analysis using Hugging Face
# ============================================================

# Install required library
!pip install transformers -q

from transformers import pipeline

# --- Load RoBERTa Sentiment Pipeline ---
print("Loading RoBERTa model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

# --- Test Sentences ---
test_sentences = [
    "I love this product, it works great!",
    "The service was terrible and I am very disappointed.",
    "The weather is nice today.",
    "I cannot believe how bad this is!"
]

# Label mapping for this model
label_map = {"positive": "POSITIVE", "neutral": "NEUTRAL", "negative": "NEGATIVE"}

print("\n--- RoBERTa Predictions ---")
print(f"{'Input':<50} {'Label':<12} {'Score':<8}")
print("-" * 75)
for sentence in test_sentences:
    result = sentiment_pipeline(sentence)[0]
    label = label_map.get(result['label'].lower(), result['label'].upper())
    print(f"{sentence[:48]:<50} {label:<12} {result['score']:.4f}")